In [16]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator
from langchain_google_genai import ChatGoogleGenerativeAI

In [17]:
load_dotenv()

True

In [18]:
# 1. Initialize the Gemini model (replaces ChatOpenAI)
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [19]:
# 2. Define the Structured Output Schema
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [20]:
# Bind the schema to the Gemini model
structured_model = model.with_structured_output(EvaluationSchema)

In [21]:
# 3. Define the Graph State
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    # operator.add acts as a reducer to merge lists from parallel nodes
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [22]:
# 4. Define the Parallel Nodes
def evaluate_language(state: EssayState):
    prompt = (
        "Evaluate the language quality of the following essay and provide "
        f"feedback and assign a score out of 10 \n {state['essay']}"
    )
    output = structured_model.invoke(prompt)
    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score],
    }

In [23]:
def evaluate_analysis(state: EssayState):
    prompt = (
        "Evaluate the depth of analysis of the following essay and provide "
        f"feedback and assign a score out of 10 \n {state['essay']}"
    )
    output = structured_model.invoke(prompt)
    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score],
    }

In [24]:
def evaluate_thought(state: EssayState):
    prompt = (
        "Evaluate the clarity of thought of the following essay and provide "
        f"feedback and assign a score out of 10 \n {state['essay']}"
    )
    output = structured_model.invoke(prompt)
    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score],
    }

In [25]:
# 5. Define the Final Aggregation Node
def final_evaluation(state: EssayState):
    # Generate summarized feedback using the standard (non-structured) Gemini model
    prompt = (
        f"Based on the following feedbacks create a summarized feedback \n "
        f"language feedback: {state['language_feedback']} \n "
        f"depth of analysis feedback: {state['analysis_feedback']} \n "
        f"clarity of thought feedback: {state['clarity_feedback']}"
    )
    overall_feedback = model.invoke(prompt).content

    # Calculate average score
    avg_score = sum(state["individual_scores"]) / len(state["individual_scores"])

    return {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score,
    }




In [26]:
# 6. Build and Compile the Graph
graph = StateGraph(EssayState)

# Add nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Add parallel fan-out edges from START
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

# Add fan-in edges to final_evaluation
graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()

In [27]:
# 7. Run the Workflow
if __name__ == "__main__":
    sample_essay = """The United States in the Age of AI: The Infrastructure and Capital Superpower
    As the world navigates the maturing landscape of artificial intelligence, the United States stands as..."""

    initial_state = {"essay": sample_essay}
    result = workflow.invoke(initial_state)

    print("Individual Scores:", result["individual_scores"])
    print("Average Score:", result["avg_score"])
    print("\nOverall Feedback:\n", result["overall_feedback"])

Individual Scores: [3, 9, 7]
Average Score: 6.333333333333333

Overall Feedback:
 [{'type': 'text', 'text': "Here is a summarized synthesis of the feedback:\n\n**Summary of Feedback:**\nThe provided introduction establishes a strong, articulate, and academically sophisticated foundation, effectively setting up a compelling thesis regarding the United States' role as an infrastructure and capital superpower in the age of AI. However, because only the opening was provided, a full evaluation of the depth of analysis and sustained clarity is not yet possible. To achieve a higher score, the subsequent paragraphs must maintain this high level of vocabulary and clarity, avoid overly dense transitions, and rigorously support the core premise with concrete evidence, specific technological and economic factors, and a well-supported conclusion.", 'extras': {'signature': 'EpYBCpMBAWkUfROxnvue74LRWwqO+PFVuuebyF2BGaxxf2FD8URe0GS5+QXrvc/k+sadN0meKXURyJvef65/8AHE3yfUZWUS8AJ04LPutF3kwDwFyPOLKAkLIbxN2fC